# v6 overlap40：冻结 Gated 后训练 CMCR

加载已经验证的 Gated batch4 checkpoint，冻结全部母模型参数，只训练零初始化 CMCR 残差。epoch 0 先保存原始 Gated 指标作为保底；最多40轮、连续8轮无提升早停；不访问 Test。

## 1. 只需填写基础 checkpoint 路径


In [ ]:
from pathlib import Path

# 先把 gated_boundary_resnet50_bw0_batch4/best_model.pth 上传为 Kaggle Dataset，
# Add Input 后把下面路径改成右侧数据面板显示的实际路径。
BASE_CHECKPOINT = Path('/kaggle/input/REPLACE_WITH_GATED_CHECKPOINT_DATASET/best_model.pth')

CONFIG_FILE = 'v6_overlap40_frozen_gated_cmcr_batch4_seed42.json'
REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'


## 2. 环境、代码与配置


In [ ]:
import hashlib, importlib.metadata, importlib.util, json, shutil, subprocess, sys

REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
import torch
assert torch.cuda.is_available(), '请在 Notebook settings 中开启 GPU'
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
config = json.loads((PROJECT_DIR / 'configs' / CONFIG_FILE).read_text(encoding='utf-8'))
assert config['module'] == 'gated_cmcr' and config['freeze_base'] is True
assert config['batch_size'] == 4 and config['accum_steps'] == 1
assert config['automatic_test_evaluation'] is False
assert BASE_CHECKPOINT.is_file(), f'基础 checkpoint 不存在，请修改第1节路径: {BASE_CHECKPOINT}'
checkpoint_hash = hashlib.sha256(BASE_CHECKPOINT.read_bytes()).hexdigest()
assert checkpoint_hash == config['expected_init_checkpoint_sha256'], (checkpoint_hash, config['expected_init_checkpoint_sha256'])
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', commit)
print('Base checkpoint:', BASE_CHECKPOINT)
print('Base SHA-256:', checkpoint_hash)
print(json.dumps(config, ensure_ascii=False, indent=2))


## 3. 核验冻结数据


In [ ]:
import numpy as np
existing = [Path(path) for path in config['data_candidates'] if Path(path).is_dir()]
assert existing, '没有找到 overlap40 数据集，请 Add Input: yuanssy/datav6-overlap40'
DATA_ROOT = existing[0]
for split, expected in config['expected_tiles'].items():
    images = sorted([*(DATA_ROOT / split / 'image').glob('*.tif'), *(DATA_ROOT / split / 'image').glob('*.tiff')])
    masks = sorted([*(DATA_ROOT / split / 'mask').glob('*.tif'), *(DATA_ROOT / split / 'mask').glob('*.tiff')])
    assert len(images) == len(masks) == expected, (split, len(images), len(masks))
    assert {p.stem for p in images} == {p.stem for p in masks}
for name, expected_hash in config['expected_metadata_sha256'].items():
    actual_hash = hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (name, actual_hash, expected_hash)
stats = json.loads((DATA_ROOT / 'normalization_stats.json').read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], config['expected_mean'], rtol=0, atol=1e-12)
assert np.allclose(stats['std'], config['expected_std'], rtol=0, atol=1e-12)
print('Data:', DATA_ROOT, '核验通过')


## 4. 冻结状态与 batch4 显存冒烟


In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from models.module_models import build_module_model
from train_module_experiment import ExperimentLoss, load_frozen_cmcr_base, model_outputs
smoke_model = build_module_model('gated_cmcr', encoder_weights=None).cuda()
report = load_frozen_cmcr_base(smoke_model, str(BASE_CHECKPOINT), torch.device('cuda'))
trainable = [(name, p) for name, p in smoke_model.named_parameters() if p.requires_grad]
assert trainable and all(name.startswith('cmcr.') for name, _ in trainable)
assert sum(p.numel() for _, p in trainable) == 42133
smoke_model.eval(); smoke_model.cmcr.train()
criterion = ExperimentLoss('gated_cmcr', 0.0).cuda()
optimizer = torch.optim.AdamW([p for _, p in trainable], lr=config['learning_rate'])
scaler = torch.amp.GradScaler('cuda')
x = torch.randn(4, 5, 512, 512, device='cuda')
labels = torch.randint(0, 5, (4, 512, 512), device='cuda')
with torch.amp.autocast('cuda'):
    logits, boundary_logits = model_outputs(smoke_model, x, True)
    loss, _ = criterion(logits, labels, boundary_logits)
scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
assert logits.shape == (4, 5, 512, 512)
print('Missing CMCR keys:', len(report['missing_keys']))
print('Trainable parameters:', sum(p.numel() for _, p in trainable))
print('Smoke loss:', float(loss))
del smoke_model, criterion, optimizer, scaler, x, labels, logits, boundary_logits, loss
torch.cuda.empty_cache()


## 5. 正式训练（epoch 0保底，早停）


In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
assert not result_dir.exists(), f'结果目录已存在: {result_dir}'
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_module_experiment.py'),
    '--module', config['module'], '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT), '--run-name', config['run_name'],
    '--seed', str(config['seed']), '--epochs', str(config['epochs']),
    '--batch-size', str(config['batch_size']), '--accum-steps', str(config['accum_steps']),
    '--num-workers', str(config['num_workers']), '--learning-rate', str(config['learning_rate']),
    '--boundary-weight', str(config['boundary_weight']), '--encoder-weights', config['encoder_weights'],
    '--init-checkpoint', str(BASE_CHECKPOINT), '--freeze-base',
    '--early-stopping-patience', str(config['early_stopping_patience']),
]
print(' '.join(command))
subprocess.check_call(command, cwd=PROJECT_DIR)


## 6. 查看并打包结果


In [ ]:
result = json.loads((result_dir / 'metrics.json').read_text(encoding='utf-8'))
assert result['test_evaluated'] is False
print(json.dumps(result, ensure_ascii=False, indent=2))
archive_path = shutil.make_archive(str(OUTPUT_ROOT / result_dir.name), 'zip', root_dir=result_dir)
print('请下载:', archive_path)
